<a href="https://colab.research.google.com/github/Dionruns/MSST-WebUI/blob/main/webUI_for_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music-Source-Separation-Training-Inference-Webui For Google Colab

LICENSE: AGPL-3.0

For personal entertainment and non-commercial use only. Prohibited from use in content related to blood, gore, violence, sex, politics.<br>
Author: [Github@KitsuneX07](https://github.com/KitsuneX07) | [Github@SUC-DriverOld](https://github.com/SUC-DriverOld) | Gradio theme: [Gradio Theme](https://huggingface.co/spaces/NoCrypt/miku)<br>
Github: [https://github.com/SUC-DriverOld/MSST-WebUI](https://github.com/SUC-DriverOld/MSST-WebUI)

## Introduction

This is a webUI for [Music-Source-Separation-Training (MSST)](https://github.com/ZFTurbo/Music-Source-Separation-Training), which is a repository for training models for music source separation. You can use this webUI to infer the MSST model and VR Models, and the preset process page allows you to customize the processing flow yourself. You can install models in the "Install Models" interface. If you have downloaded [Ultimate Vocal Remover (UVR)](https://github.com/Anjok07/ultimatevocalremovergui) before, you do not need to download VR Models again. You can go to the "Settings" page and directly select your UVR5 model folder. We also provide some convenient tools in the WebUI such as [Singing-Oriented MIDI Extractor (SOME)](https://github.com/openvpi/SOME/), advanced ensemble mode, and more.

In [1]:
# @title Clone repository and install requirements
# @markdown # Clone repository and install requirements
# @markdown
# @markdown ### You may need to rerun this cell for a few times
# @markdown

# Check GPU
!nvidia-smi

# Clone the repository
%cd /content
!git clone https://github.com/SUC-DriverOld/MSST-WebUI
%cd /content/MSST-WebUI
!git checkout ccbce58

# Install requirements
!sudo apt install ffmpeg
%pip install uv
!uv sync

Mon Sep 28 16:36:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title [Optional] Download SOME weights if needed
# @markdown # Download SOME weights if needed
# @markdown
# @markdown ### If you want to use "Vocals to MIDI", download the weights.
# @markdown
# @markdown ### "Vocals to MIDI" use project: [SOME](https://github.com/openvpi/SOME/).
# @markdown

!wget https://huggingface.co/Sucial/MSST-WebUI/resolve/main/SOME_weights/model_steps_64000_simplified.ckpt -O /content/MSST-WebUI/tools/SOME_weights/model_steps_64000_simplified.ckpt

--2026-09-28 16:45:02--  https://huggingface.co/Sucial/MSST-WebUI/resolve/main/SOME_weights/model_steps_64000_simplified.ckpt
Resolving huggingface.co (huggingface.co)... 13.33.45.68, 13.33.45.10, 13.33.45.84, ...
Connecting to huggingface.co (huggingface.co)|13.33.45.68|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66fe2159df07d774c5f53db7/7733568ce8d7959c489250ee673c7996dc6cb4dc076eec154c30b77550370223?X-Xet-Cas-Uid=public&user_id=public&xip=80z7YpmiY1o&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model_steps_64000_simplified.ckpt%3B+filename%3D%22model_steps_64000_simplified.ckpt%22%3B&Expires=1790617502&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZmZTIxNTlkZjA3ZDc3NGM1ZjUzZGI3Lzc3MzM1NjhjZThkNzk1OWM0ODkyNTBlZTY3M2M3OTk2ZGM2Y2I0ZGMwNzZlZWMxNTRjMzBiNzc1NTAzNzAyMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZ4aXA9ODB6N1lwbWlZMW8mcmVzcG9uc

In [ ]:
# @title [Optional] Pack and upload your music files to your google drive
# @markdown # Pack and upload your music files to your google drive
# @markdown
# @markdown Ensure that there are no nested folders in your zip file, but rather direct audio files.
# @markdown

from google.colab import drive

drive.mount("/content/drive")
# @markdown Directory where **your zip file** located in, dont miss the slash at the end.
data_dir = "/content/drive/MyDrive/MSST-WebUI/"  # @param {type:"string"}
# @markdown Filename of **your zip file**, for example: "inputs.zip"
zip_filename = "input.zip"  # @param {type:"string"}
ZIP_PATH = data_dir + zip_filename
# @markdown You need to package the audio directly into a zip file, **DO NOT** nest folders in the zip file!
# @markdown The compressed file will be extracted to the `input` directory. If there are files in your `input` directory, they will be deleted. Please backup in advance.
# @markdown

!rm -rf /content/MSST-WebUI/input
!mkdir -p /content/MSST-WebUI/input
!unzip -od /content/MSST-WebUI/input {ZIP_PATH}

In [ ]:
# @title [Optional] Save results to google drive
# @markdown # Save results to google drive
# @markdown
# @markdown The results audio files in the output directory will be compressed into a zip file and saved to your google drive.
# @markdown

from google.colab import drive

drive.mount("/content/drive")
# @markdown Path to save the zip file, dont miss the slash at the end.
output_dir = "/content/drive/MyDrive/MSST-WebUI/"  # @param {type:"string"}
# @markdown Filename of the zip file, for example: "Outputs.zip"
zip_filename = "Outputs.zip"  # @param {type:"string"}
ZIP_PATH = output_dir + zip_filename

!mkdir -p {output_dir}
!zip -r "{zip_filename}" /content/MSST-WebUI/results
!cp -vr "{zip_filename}" {output_dir}

In [ ]:
# @title [Optional] Delete all result audio files
# @markdown # Click to delete all result audio files
# @markdown
# @markdown ### Warning: Unrecoverable, please operate with caution
# @markdown

!rm -rf /content/MSST-WebUI/results
!mkdir -p /content/MSST-WebUI/results

In [ ]:
# @title [Optional] Delete all input audio files
# @markdown # Click to delete all input audio files
# @markdown
# @markdown ### Warning: Unrecoverable, please operate with caution
# @markdown

!rm -rf /content/MSST-WebUI/input
!mkdir -p /content/MSST-WebUI/input

In [ ]:
# @title Initialize and start WebUI
# @markdown # Initialize and start WebUI
# @markdown
# @markdown After running this cell, you can use your own device to connect to WebUI. The public link will be displayed below.
# @markdown
# @markdown If you want to modify the inference parameters of the preset process, edit the `data/webui_config.json` file.
# @markdown

%cd /content/MSST-WebUI

LANGUAGE = "简体中文"  # @param ["Auto", "English", "简体中文", "繁體中文", "日本語", "😊", "한국어"]
MODEL_DOWNLOAD_LINK = "huggingface.co"  # @param ["Auto", "huggingface.co", "hf-mirror.com"]
DEBUG = False  # @param {type:"boolean"}

language_dict = {"Auto": "Auto", "简体中文": "zh_CN", "繁體中文": "zh_TW", "English": "en_US", "日本語": "ja_JP", "😊": "emoji", "한국어": "ko_KR"}
language = language_dict[LANGUAGE]
debug = "--debug" if DEBUG else ""

# Start WebUI
!MPLBACKEND=Agg uv run webUI.py --use_cloud --share --language {language} --model_download_link {MODEL_DOWNLOAD_LINK} {debug}

/content/MSST-WebUI
16:47:50.514 [INFO]    [webui/utils.py:99] Console log level set to [INFO]    
16:47:50.670 [INFO]    [tools/webUI_for_clouds/webUI_for_clouds.py:211] 检测到CUDA, 设备信息: {'cuda0': '0: Tesla T4'}
16:47:50.670 [INFO]    [tools/webUI_for_clouds/webUI_for_clouds.py:222] WebUI Version: 1.7.0 v2, System: Linux, Machine: x86_64
/content/MSST-WebUI/.venv/lib/python3.10/site-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.38.1, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
/content/MSST-WebUI/.venv/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
Running on local URL:  http://127.0.0.1:7860
Running on public URL: htt